# 1. Data processing

Builds the analysis cohorts, derived features and patient-level splits from the
raw workbooks. No predictive model is trained here.

Outcome convention: `y_event = 1` means cardiac death within the horizon,
`y_event = 0` means observed event-free through the horizon. Patients censored
before the horizon, and non-cardiac deaths before the horizon, are excluded from
the strict classification cohort. The time-to-event cohort retains all patients with positive observed
risk time and is used for the survival analyses. Two alive patients
with zero recorded follow-up are excluded from this cohort; under the
fixed-horizon definition they are also excluded from the strict
cohorts as censored before the horizon.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

import config
import utils

utils.ensure_dirs()
utils.set_seed()
print("profile:", config.PROFILE, "| horizons:", config.HORIZONS, "| seed:", config.SEED)

profile: full | horizons: (7, 10) | seed: 42


## 1.1 Load and merge the raw workbooks

In [2]:
def read_workbook(path):
    frame = pd.read_excel(path, engine="openpyxl")
    frame.columns = [c.strip() for c in frame.columns]
    return frame

raw = read_workbook(config.RAW_CLINICAL)
n_raw_rows = len(raw)
raw = raw.dropna(subset=["Number"]).copy()
raw["Number"] = raw["Number"].astype(int)

enrolment = read_workbook(config.RAW_ENROLMENT)
enrolment["Number"] = enrolment["Number"].astype(int)
enrolment = enrolment[["Number", "Data prelievo"]]

# Creatinine is read only to document its missingness; it is not a feature.
labs = read_workbook(config.RAW_LABS)
labs["Number"] = labs["Number"].astype(int)
labs = labs[["Number", "Creatinina"]]

print(f"raw_data rows: {n_raw_rows} -> {len(raw)} after dropping a missing identifier")
for name, frame in [("enrolment", enrolment), ("labs", labs)]:
    print(f"{name}: {len(frame)} rows, duplicated keys: {int(frame['Number'].duplicated().sum())}")

raw_data rows: 8144 -> 8065 after dropping a missing identifier
enrolment: 8065 rows, duplicated keys: 0
labs: 8065 rows, duplicated keys: 0


In [3]:
# validate="m:1" fails loudly if an auxiliary workbook ever gains a duplicate
# key, which would silently fan one patient into several rows.
df = raw.merge(enrolment, on="Number", how="left", validate="m:1")
df = df.merge(labs, on="Number", how="left", validate="m:1")

collisions = [c for c in df.columns if c.endswith("_x") or c.endswith("_y")]
if collisions:
    print("column collisions introduced by the merge:", collisions)

df = df.rename(columns=config.RENAME)
utils.check_unique_ids(df)
print(f"merged: {len(df)} rows, {df.shape[1]} columns, unique patient identifiers")

merged: 8065 rows, 85 columns, unique patient identifiers


## 1.2 Patient-level corrections

The raw workbooks are never modified. Corrections are declared in
`config.DATA_CORRECTIONS` and logged with their before and after values.

In [4]:
records = []
for correction in config.DATA_CORRECTIONS:
    mask = df[config.ID_COL] == correction["patient_id"]
    if not mask.any():
        records.append({**{k: correction[k] for k in ("patient_id", "issue", "rationale")},
                        "status": "patient not found"})
        continue
    columns = list(correction["updates"])
    before = df.loc[mask, columns].iloc[0].to_dict()
    for column, value in correction["updates"].items():
        df.loc[mask, column] = value
    after = df.loc[mask, columns].iloc[0].to_dict()
    records.append({
        "patient_id": correction["patient_id"],
        "issue": correction["issue"],
        "status": "applied",
        "before": before,
        "after": after,
        "rationale": correction["rationale"],
    })

corrections = pd.DataFrame(records)
utils.save_result(corrections, "data_corrections")
corrections

,patient_id,issue,status,before,after,rationale
0,6850,Conflicting thyroid flags: SCH and Hyperthyroi...,applied,"{'SCH': 1.0, 'Hyperthyroid': 1.0}","{'SCH': 1.0, 'Hyperthyroid': 0.0}",Clinical review confirmed the subclinical hypo...
1,7286,Death date present but the all-cause mortality...,applied,{'death_any': 0.0},{'death_any': 1.0},A populated death date must be consistent with...


## 1.3 Follow-up duration and events

In [5]:
for column in config.DATE_COLS:
    df[column] = pd.to_datetime(df[column], errors="coerce", dayfirst=True)

end_date = df["death_date"].fillna(df["followup_date"])
df["followup_days"] = (end_date - df["enrolment_date"]).dt.days.clip(lower=0)
df["followup_years"] = df["followup_days"] / config.DAYS_PER_YEAR

df["event_cardiac"] = (df["death_cardiac"] == 1).astype(int)
df["event_any_death"] = (df["death_any"] == 1).astype(int)
df["event_noncardiac"] = ((df["event_any_death"] == 1) &
                          (df["event_cardiac"] == 0)).astype(int)

# A same-day cardiac death is a real event but has no risk time, so survival
# models see one day instead of zero. The classification horizon uses the
# original duration.
same_day = (df["event_cardiac"] == 1) & (df["followup_days"] == 0)
df["survival_time_days"] = df["followup_days"].where(~same_day, 1)
df["survival_time_years"] = df["survival_time_days"] / config.DAYS_PER_YEAR

print(f"cardiac deaths: {int(df['event_cardiac'].sum())}")
print(f"non-cardiac deaths: {int(df['event_noncardiac'].sum())}")
print(f"same-day cardiac deaths shifted to one day: {int(same_day.sum())}")
print(f"median follow-up: {df['followup_years'].median():.2f} years")

cardiac deaths: 1008
non-cardiac deaths: 1399
same-day cardiac deaths shifted to one day: 1
median follow-up: 6.36 years


## 1.4 Derived thyroid features

Missing thyroid status stays missing. Encoding an unknown state as euthyroid
would silently impute a normal value and would also change which patients each
feature set can use.

In [6]:
state_columns = ["Euthyroid"] + config.THYROID_STATES
present = df[state_columns].notna().all(axis=1)
active = df.loc[present, state_columns].sum(axis=1)
print(f"patients with a complete thyroid state block: {int(present.sum())} / {len(df)}")
print(f"state blocks that are not mutually exclusive: {int((active > 1).sum())}")
print(f"state blocks with no state set: {int((active == 0).sum())}")

df["fT3_fT4_ratio"] = np.where(df["fT4"].notna() & (df["fT4"] != 0),
                               df["fT3"] / df["fT4"], np.nan)
df["thyroid_abnormal"] = np.where(df["Euthyroid"].isna(), np.nan,
                                  (df["Euthyroid"] == 0).astype(float))

patients with a complete thyroid state block: 8065 / 8065
state blocks that are not mutually exclusive: 0
state blocks with no state set: 0


## 1.5 Feature sets

In [7]:
rows = []
for name, features in config.FEATURE_SETS.items():
    missing = [f for f in features if f not in df.columns]
    if missing:
        raise KeyError(f"Feature set {name} requires missing columns: {missing}")
    rows.append({
        "feature_set": name,
        "n_features": len(features),
        "role": ("primary baseline" if name == config.PRIMARY_BASELINE else
                 "primary thyroid" if name == config.PRIMARY_THYROID else "secondary"),
        "rationale": config.FEATURE_SET_RATIONALE[name],
    })
feature_sets = pd.DataFrame(rows)
utils.save_result(feature_sets, "feature_sets")
feature_sets

,feature_set,n_features,role,rationale
0,CV17,17,primary baseline,Cardiovascular predictors alone (baseline).
1,CV17_THY_CONT,20,primary thyroid,Do the continuous thyroid biomarkers add value?
2,CV17_THY_STATES,22,secondary,Do the clinical thyroid categories add value?
3,CV17_THY_CONT_STATES,25,secondary,Do the categories add anything beyond the cont...
4,CV17_THY_CONT_RATIO,21,secondary,Does the fT3/fT4 ratio add value?


In [8]:
used = sorted({f for features in config.FEATURE_SETS.values() for f in features})
missingness = df[used + ["creatinine"]].isna().sum()
missingness = (missingness[missingness > 0] / len(df) * 100).round(2)
print("missing values by column, percent of the full cohort")
print(missingness if len(missingness) else "none")

# Creatinine is the paper's eighteenth predictor. It is excluded here because of
# its missingness, so the baseline holds seventeen cardiovascular variables.
print(f"\ncreatinine missing: {df['creatinine'].isna().mean() * 100:.1f} percent")

missing values by column, percent of the full cohort
creatinine    7.39
dtype: float64

creatinine missing: 7.4 percent


## 1.6 Cohorts

In [9]:
utils.write_frame(df, config.COHORT_FULL)

survival_cohort = df[(df["survival_time_days"] > 0)].copy()
utils.write_frame(survival_cohort, config.COHORT_SURVIVAL)
print(f"full cohort: {len(df)}")
print(f"survival cohort: {len(survival_cohort)}, "
      f"cardiac events {int(survival_cohort['event_cardiac'].sum())}, "
      f"non-cardiac deaths {int(survival_cohort['event_noncardiac'].sum())}")

full cohort: 8065
survival cohort: 8063, cardiac events 1008, non-cardiac deaths 1399


In [10]:
flow = []
strict_cohorts = {}
for horizon in config.HORIZONS:
    limit = horizon * config.DAYS_PER_YEAR
    event = (df["event_cardiac"] == 1) & (df["followup_days"] <= limit)
    event_free = (df["followup_days"] >= limit) & (~event)
    censored_alive = (~event) & (~event_free) & (df["event_any_death"] == 0)
    noncardiac_early = (df["event_noncardiac"] == 1) & (df["followup_days"] < limit)

    strict = df[event | event_free].copy()
    strict[config.TARGET] = event[event | event_free].astype(int).to_numpy()
    strict_cohorts[horizon] = strict
    utils.write_frame(strict, str(config.COHORT_STRICT).format(horizon=horizon))

    flow.append({
        "horizon": horizon,
        "full_cohort": len(df),
        "events_within_horizon": int(event.sum()),
        "event_free_through_horizon": int(event_free.sum()),
        "excluded_censored_before_horizon": int(censored_alive.sum()),
        "excluded_noncardiac_death_before_horizon": int(noncardiac_early.sum()),
        "strict_cohort": len(strict),
        "prevalence": round(float(strict[config.TARGET].mean()), 4),
    })

cohort_flow = pd.DataFrame(flow)
utils.save_result(cohort_flow, "cohort_flow")
cohort_flow

,horizon,full_cohort,events_within_horizon,event_free_through_horizon,excluded_censored_before_horizon,excluded_noncardiac_death_before_horizon,strict_cohort,prevalence
0,7,8065,843,3547,2628,1047,4390,0.192
1,10,8065,962,1652,4179,1272,2614,0.368


In [11]:
# The defining invariant of the strict cohort: nobody is called event-free
# unless they were actually observed through the horizon.
for horizon, strict in strict_cohorts.items():
    limit = horizon * config.DAYS_PER_YEAR
    free = strict[strict[config.TARGET] == 0]
    assert (free["followup_days"] >= limit).all(), horizon
    events = strict[strict[config.TARGET] == 1]
    assert (events["followup_days"] <= limit).all(), horizon
print("strict cohort invariants hold for every horizon")

strict cohort invariants hold for every horizon


## 1.7 Cohort description

Descriptive comparison of the two outcome groups in the strict cohort, in the
style of the paper's baseline table. Continuous variables use the Mann-Whitney
test and binary variables the chi-squared test. These are unadjusted
descriptions, not evidence of incremental value.

In [12]:
from scipy.stats import chi2_contingency, mannwhitneyu

reference = strict_cohorts[config.HORIZONS[0]]
rows = []
for feature in sorted({f for fs in config.FEATURE_SETS.values() for f in fs}):
    values = reference[feature]
    event = reference[config.TARGET] == 1
    if feature in config.CONTINUOUS_FEATURES:
        a = values[event].dropna()
        b = values[~event].dropna()
        p = mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan
        rows.append({"variable": feature, "type": "continuous",
                     "event_summary": f"{a.mean():.2f} ({a.std():.2f})",
                     "event_free_summary": f"{b.mean():.2f} ({b.std():.2f})",
                     "test": "Mann-Whitney", "p_value": p})
    else:
        table = pd.crosstab(values, event)
        p = chi2_contingency(table).pvalue if table.shape == (2, 2) else np.nan
        rows.append({"variable": feature, "type": "binary",
                     "event_summary": f"{100 * values[event].mean():.1f} percent",
                     "event_free_summary": f"{100 * values[~event].mean():.1f} percent",
                     "test": "chi-squared", "p_value": p})

description = pd.DataFrame(rows).sort_values("p_value")
utils.save_result(description, "cohort_description")
description.round(5)

,variable,type,event_summary,event_free_summary,test,p_value
2,Age,continuous,75.12 (10.34),64.99 (12.27),Mann-Whitney,0.00000
12,LVEF,continuous,41.66 (15.05),53.26 (10.57),Mann-Whitney,0.00000
14,PostIsch_DCM,binary,26.2 percent,5.6 percent,chi-squared,0.00000
24,fT3_fT4_ratio,continuous,0.20 (0.35),0.21 (0.17),Mann-Whitney,0.00000
25,fT4,continuous,13.02 (3.96),12.11 (5.06),Mann-Whitney,0.00000
5,Diabetes,binary,33.9 percent,18.4 percent,chi-squared,0.00000
0,AFib,binary,30.2 percent,16.2 percent,chi-squared,0.00000
6,Dyslipidemia,binary,53.1 percent,69.5 percent,chi-squared,0.00000
15,Previous_CABG,binary,15.8 percent,6.8 percent,chi-squared,0.00000
16,Previous_MI,binary,39.3 percent,25.3 percent,chi-squared,0.00000


In [13]:
# The reference study enrolled only patients meeting its definition of
# ischemic heart disease; no additional diagnostic filter is applied during
# data preparation here. Quantify, per cohort, how many patients satisfy at
# least one condition of that definition through the recorded fields that
# correspond to it: stenosed vessels, acute and previous myocardial
# infarction, bypass surgery, coronary intervention and post-ischemic
# dilated cardiomyopathy.
IHD_CONDITIONS = ["Previous_MI", "Acute_MI", "Previous_CABG",
                  "Previous_PCI", "PostIsch_DCM"]


def meets_ihd_field_proxy(frame):
    met = frame["Vessels"].fillna(0) >= 1
    for column in IHD_CONDITIONS:
        met = met | (frame[column].fillna(0) > 0)
    return met


rows = []
cohorts = [("full", df), ("survival", survival_cohort)]
cohorts += [(f"strict_h{horizon}", strict)
            for horizon, strict in strict_cohorts.items()]
for label, frame in cohorts:
    met = meets_ihd_field_proxy(frame)
    rows.append({"cohort": label, "n": len(frame),
                 "n_flagged_by_ihd_field_proxy": int(met.sum()),
                 "share_flagged_by_ihd_field_proxy": round(float(met.mean()), 4)})

diagnosis_composition = pd.DataFrame(rows)
utils.save_result(diagnosis_composition, "cohort_diagnosis_composition")
diagnosis_composition


,cohort,n,n_flagged_by_ihd_field_proxy,share_flagged_by_ihd_field_proxy
0,full,8065,4261,0.5283
1,survival,8063,4261,0.5285
2,strict_h7,4390,2409,0.5487
3,strict_h10,2614,1497,0.5727


## 1.8 Splits

One stratified 60/20/20 split per horizon, stored as patient identifiers so that
every feature set, model and comparison provably uses the same patients.

In [14]:
splits_by_horizon = {}
summary = []
for horizon, strict in strict_cohorts.items():
    splits = utils.make_splits(strict)
    utils.assert_disjoint(splits)
    splits_by_horizon[f"h{horizon}"] = splits

    lookup = strict.set_index(config.ID_COL)[config.TARGET]
    for part, members in splits.items():
        y_part = lookup.loc[members]
        summary.append({
            "horizon": horizon, "split": part, "n": len(members),
            "n_events": int(y_part.sum()),
            "prevalence": round(float(y_part.mean()), 4),
        })

utils.save_splits(splits_by_horizon)
split_summary = pd.DataFrame(summary)
utils.save_result(split_summary, "split_summary")
split_summary

,horizon,split,n,n_events,prevalence
0,7,train,2634,506,0.1921
1,7,valid,878,169,0.1925
2,7,test,878,168,0.1913
3,10,train,1568,577,0.3680
4,10,valid,523,193,0.3690
5,10,test,523,192,0.3671


In [15]:
for horizon, strict in strict_cohorts.items():
    splits = utils.load_splits(horizon)
    covered = sorted(splits["train"] + splits["valid"] + splits["test"])
    assert covered == sorted(strict[config.ID_COL].astype(int)), horizon
    for feature_set in config.FEATURE_SETS:
        _, _, ids = utils.extract_xy(strict, feature_set)
        utils.assert_same_patients(strict[config.ID_COL].to_numpy(), ids)
print("splits cover the strict cohort exactly and every feature set uses the same patients")

splits cover the strict cohort exactly and every feature set uses the same patients


In [16]:
utils.write_manifest(
    config.RESULTS_DIR / "data_process_manifest.json",
    stage="1_data_process",
    n_full=len(df),
    n_survival=len(survival_cohort),
    strict_cohorts={int(h): int(len(s)) for h, s in strict_cohorts.items()},
    corrections_applied=int((corrections["status"] == "applied").sum()),
    data_fingerprint=utils.data_fingerprint(df, used + [config.TARGET]
                                            if config.TARGET in df.columns else used))
print("stage complete")

stage complete
